# Headline Preprocessing

Build a feature dataframe from parsed headline JSON:
- supports `title_template_slot_extractions_parser.json`, `title_template_slot_extractions_parser_public.json`, and `title_template_slot_extractions_parser_private.json`
- one-hot encode `template_index`, `company`, `sector`, and `region`
- keep extracted number fields (e.g. `dollar`)
- add `ThisCompany` and `ThisSector` flags using the **session-specific entity with highest average forward 5-bar log-return volatility**


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Choose which parsed headline file to preprocess
DATASET = 'train'  # 'train', 'public', or 'private'

INPUT_OPTIONS = {
    'train': Path('title_template_slot_extractions_parser.json'),
    'public': Path('title_template_slot_extractions_parser_public.json'),
    'private': Path('title_template_slot_extractions_parser_private.json'),
}
if DATASET not in INPUT_OPTIONS:
    raise ValueError(f"DATASET must be one of {list(INPUT_OPTIONS)}")

INPUT_PATH = INPUT_OPTIONS[DATASET]
OUTPUT_PARQUET = Path(f'headline_features_{DATASET}.parquet')
OUTPUT_CSV = Path(f'headline_features_{DATASET}.csv')

# Use all available bars parquet files so train/public/private parser files are supported
DATA_DIR = Path('hrt-eth-zurich-datathon-2026/data')
BARS_PATHS = sorted(DATA_DIR.glob('bars_*.parquet'))
VOL_K = 5
MIN_ENTITY_ARTICLES = 3  # minimum headlines per (session, company/sector) to be eligible

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'{INPUT_PATH.resolve()} not found.')
if not BARS_PATHS:
    raise FileNotFoundError(f'No bars_*.parquet files found under {DATA_DIR.resolve()}')

print('Dataset:', DATASET)
print('Input:', INPUT_PATH.resolve())
print('Output parquet:', OUTPUT_PARQUET.resolve())
print('Output csv:', OUTPUT_CSV.resolve())
print('Bars files:', [p.name for p in BARS_PATHS])
print('Volatility horizon K:', VOL_K)
print('Min entity articles:', MIN_ENTITY_ARTICLES)


In [ ]:
records = json.loads(INPUT_PATH.read_text(encoding='utf-8'))
df = pd.DataFrame(records)

required_cols = [
    'title', 'template_index', 'dollar', 'company', 'sector',
    'row_index', 'session', 'bar_ix'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = df.sort_values('row_index').reset_index(drop=True)
df['template_index'] = pd.to_numeric(df['template_index'], errors='coerce').fillna(-1).astype(int)
df['session'] = pd.to_numeric(df['session'], errors='coerce').astype('Int64')
df['bar_ix'] = pd.to_numeric(df['bar_ix'], errors='coerce').astype('Int64')

# Normalize string columns for downstream rules
for c in ['company', 'sector', 'dollar', 'percentage', 'region', 'template_text']:
    if c in df.columns:
        df[c] = df[c].fillna('').astype(str).str.strip()

print('Rows:', len(df))
print('Distinct template_index:', df['template_index'].nunique())
df.head()


In [ ]:
# Session-specific ThisCompany/ThisSector by avg forward log-return volatility
bars = pd.concat([pd.read_parquet(p) for p in BARS_PATHS], ignore_index=True)
bars = bars[['session', 'bar_ix', 'close']].dropna().copy()
bars['session'] = pd.to_numeric(bars['session'], errors='coerce').astype('Int64')
bars['bar_ix'] = pd.to_numeric(bars['bar_ix'], errors='coerce').astype('Int64')
bars['close'] = pd.to_numeric(bars['close'], errors='coerce')
bars = bars.dropna(subset=['session', 'bar_ix', 'close']).copy()
bars['session'] = bars['session'].astype(int)
bars['bar_ix'] = bars['bar_ix'].astype(int)
bars = bars.sort_values(['session', 'bar_ix']).reset_index(drop=True)

session_data = {}
for s, g in bars.groupby('session', sort=False):
    g = g.sort_values('bar_ix')
    bar_ix_arr = g['bar_ix'].to_numpy(dtype=int)
    close_arr = g['close'].to_numpy(dtype=float)
    pos = {int(b): i for i, b in enumerate(bar_ix_arr)}
    session_data[int(s)] = (close_arr, pos)

def forward_log_vol(session: int, bar_ix: int, k: int = VOL_K) -> float:
    if pd.isna(session) or pd.isna(bar_ix):
        return np.nan
    data = session_data.get(int(session))
    if data is None:
        return np.nan
    close_arr, pos = data
    p = pos.get(int(bar_ix))
    if p is None:
        return np.nan
    end = min(p + k, len(close_arr) - 1)
    if end <= p:
        return np.nan
    window = close_arr[p:end + 1]
    if np.any(window <= 0):
        return np.nan
    rets = np.diff(np.log(window))
    if rets.size == 0:
        return np.nan
    return float(np.sqrt(np.mean(np.square(rets))))

df['fwd_log_vol_k5'] = [forward_log_vol(s, b, VOL_K) for s, b in zip(df['session'], df['bar_ix'])]

company_stats_all = (
    df.loc[(df['company'] != '') & df['fwd_log_vol_k5'].notna(), ['session', 'company', 'fwd_log_vol_k5']]
    .groupby(['session', 'company'])['fwd_log_vol_k5']
    .agg(['mean', 'count'])
    .reset_index()
)
sector_stats_all = (
    df.loc[(df['sector'] != '') & df['fwd_log_vol_k5'].notna(), ['session', 'sector', 'fwd_log_vol_k5']]
    .groupby(['session', 'sector'])['fwd_log_vol_k5']
    .agg(['mean', 'count'])
    .reset_index()
)

company_stats = company_stats_all[company_stats_all['count'] >= MIN_ENTITY_ARTICLES].copy()
sector_stats = sector_stats_all[sector_stats_all['count'] >= MIN_ENTITY_ARTICLES].copy()

session_top_company_df = (
    company_stats
    .sort_values(['session', 'mean', 'count', 'company'], ascending=[True, False, False, True])
    .drop_duplicates('session')
    .rename(columns={'company': 'session_top_company', 'mean': 'session_top_company_mean_vol', 'count': 'session_top_company_n'})
)
session_top_sector_df = (
    sector_stats
    .sort_values(['session', 'mean', 'count', 'sector'], ascending=[True, False, False, True])
    .drop_duplicates('session')
    .rename(columns={'sector': 'session_top_sector', 'mean': 'session_top_sector_mean_vol', 'count': 'session_top_sector_n'})
)

df = df.merge(
    session_top_company_df[['session', 'session_top_company', 'session_top_company_mean_vol', 'session_top_company_n']],
    on='session',
    how='left',
)
df = df.merge(
    session_top_sector_df[['session', 'session_top_sector', 'session_top_sector_mean_vol', 'session_top_sector_n']],
    on='session',
    how='left',
)

for c in ['session_top_company', 'session_top_sector']:
    df[c] = df[c].fillna('')

df['ThisCompany'] = (
    (df['company'] != '')
    & (df['company'].str.casefold() == df['session_top_company'].str.casefold())
).astype('int8')
df['ThisSector'] = (
    (df['sector'] != '')
    & (df['sector'].str.casefold() == df['session_top_sector'].str.casefold())
).astype('int8')

print('Eligible company candidates (count>=min):', len(company_stats), '/', len(company_stats_all))
print('Eligible sector candidates (count>=min):', len(sector_stats), '/', len(sector_stats_all))
print('Sessions with volatility-based top company:', int((df['session_top_company'] != '').groupby(df['session']).any().sum()))
print('Sessions with volatility-based top sector:', int((df['session_top_sector'] != '').groupby(df['session']).any().sum()))
print('ThisCompany positives:', int(df['ThisCompany'].sum()))
print('ThisSector positives:', int(df['ThisSector'].sum()))
print('Rows with fwd_log_vol_k5 available:', int(df['fwd_log_vol_k5'].notna().sum()))

df[[
    'session', 'company', 'session_top_company', 'session_top_company_n', 'ThisCompany',
    'sector', 'session_top_sector', 'session_top_sector_n', 'ThisSector', 'fwd_log_vol_k5'
]].head()


In [ ]:
# One-hot encodings
template_ohe = pd.get_dummies(
    df['template_index'],
    prefix='template_id',
    dtype='int8',
)

company_cat = df['company'].fillna('').astype(str).str.strip().replace('', '<NONE>')
sector_cat = df['sector'].fillna('').astype(str).str.strip().replace('', '<NONE>')
region_cat = df['region'].fillna('').astype(str).str.strip().replace('', '<NONE>')

company_ohe = pd.get_dummies(company_cat, prefix='company', dtype='int8')
sector_ohe = pd.get_dummies(sector_cat, prefix='sector', dtype='int8')
region_ohe = pd.get_dummies(region_cat, prefix='region', dtype='int8')

feature_df = pd.concat([
    df,
    template_ohe,
    company_ohe,
    sector_ohe,
    region_ohe,
], axis=1)

# Keep dollars and other extracted fields as-is; categorical fields are one-hot encoded
feature_df.to_parquet(OUTPUT_PARQUET, index=False)
feature_df.to_csv(OUTPUT_CSV, index=False)

print('Saved parquet:', OUTPUT_PARQUET.resolve())
print('Saved csv:', OUTPUT_CSV.resolve())
print('Feature shape:', feature_df.shape)
print('Template OHE cols:', template_ohe.shape[1])
print('Company OHE cols:', company_ohe.shape[1])
print('Sector OHE cols:', sector_ohe.shape[1])
print('Region OHE cols:', region_ohe.shape[1])

feature_df.head()


In [ ]:
# Optional: minimal model-ready subset
id_cols = ['row_index', 'session', 'bar_ix', 'title']
raw_cols = [
    'template_index', 'template_text', 'dollar', 'percentage', 'company', 'sector', 'region',
    'fwd_log_vol_k5', 'session_top_company', 'session_top_sector',
    'session_top_company_mean_vol', 'session_top_sector_mean_vol',
]
flag_cols = ['ThisCompany', 'ThisSector']
ohe_cols = [
    c for c in feature_df.columns
    if c.startswith(('template_id_', 'company_', 'sector_', 'region_'))
]

model_df = feature_df[id_cols + raw_cols + flag_cols + ohe_cols]
print('Model df shape:', model_df.shape)
model_df.head()
print('Prepared for dataset:', DATASET)
